# **Aula2**

In [ ]:
library(stringr)
library(dplyr)
library(purrr)

dados <- read.csv("corpus.csv", stringsAsFactors = FALSE, encoding = "UTF-8")

# CADA PARÁGRAFO É UM DOCUMENTO. Nada de str_split.
# O id original vira o nome do documento — é o que o qrels usa.
docs <- setNames(dados$texto, dados$id)

length(docs)            # tem que dar 194
head(names(docs), 3)    # "gonzaga-santos-p1" ...

[1] 194

[1] "gonzaga-santos-p1" "gonzaga-santos-p2" "gonzaga-santos-p3"

In [ ]:
tokenizar <- function(texto) {
  texto <- tolower(texto)
  unlist(strsplit(texto, "\\s+"))
}
tokens <- lapply(docs, tokenizar)
tokens[["gonzaga-santos-p1"]]

[1] "o"              "gonzaga"        "é"              "um"            
 [5] "bairro"         "nobre"          "da"             "cidade"        
 [9] "de"             "santos."        "a"              "praça"         
[13] "da"             "independência," "situada"        "no"            
[17] "gonzaga"        "é"              "um"             "ponto"         
[21] "de"             "referência"     "do"             "bairro,"       
[25] "local"          "onde"           "são"            "feitas"        
[29] "manifestações," "apresentações"  "e"              "comemorações."

In [ ]:
vocab <- sort(unique(unlist(tokens)))
length(vocab)

tdm <- sapply(tokens, function(tk) {
  as.integer(table(factor(tk, levels = vocab)))
})
rownames(tdm) <- vocab
tdm[1:5, 1:5]

[1] 4098

,gonzaga-santos-p1,gonzaga-santos-p2,gonzaga-santos-p3,gonzaga-santos-p4,gonzaga-santos-p5
-,0,0,0,0,0
"-,",0,0,0,0,0
–,0,0,0,0,0
"'tatico'),",0,0,0,0,0
"""a",0,0,0,0,0


 **O que sugere para a próxima aula?**

Para a próxima aula, é necessária uma ferramenta de ponderação textual, que reduza o peso de termos muito frequentes e pouco informativos (como as stopwords), permitindo ranquear os documentos por relevância em vez de apenas indicar se um termo está presente ou não

In [ ]:
# Matriz termo-documento

tdm <- sapply(tokens, function(tk) { #roda esse processo inteiro pra cada um dos 8 documentos dentro da lista tokens
    as.integer(table(factor(tk, levels = vocab))) #pega os tokens daquele documento (tk) e transforma num fator,
    #forçando os "níveis possíveis" a serem exatamente as 45 palavras do vocabulário completo (vocab),
    # não só as palavras que aparecem naquele documento específico.
})


rownames(tdm) <- vocab

tdm[40:50, ] # 6 primeiras linhas e com todas as colunas

,gonzaga-santos-p1,gonzaga-santos-p2,gonzaga-santos-p3,gonzaga-santos-p4,gonzaga-santos-p5,gonzaga-santos-p6,gonzaga-santos-p7,boqueirao-santos-p1,boqueirao-santos-p2,boqueirao-santos-p3,⋯,praia-grande-sao-paulo-p56,praia-grande-sao-paulo-p57,praia-grande-sao-paulo-p58,praia-grande-sao-paulo-p59,praia-grande-sao-paulo-p60,praia-grande-sao-paulo-p61,praia-grande-sao-paulo-p62,praia-grande-sao-paulo-p63,praia-grande-sao-paulo-p64,praia-grande-sao-paulo-p65
"""rancho",0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
"""rancho"",",0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
"""reação",0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
"""região",0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
"""são",0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
"""sustentou,",0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
"""torre",0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
"""vale",0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
"(013),",0,0,0,0,0,0,0,0,0,0,⋯,0,1,0,0,0,0,0,0,0,0
(0132).,0,0,0,0,0,0,0,0,0,0,⋯,1,0,0,0,0,0,0,0,0,0


In [ ]:
busca_booleana <- function(termo, tdm) {
  termo <- tolower(termo)
  if (!termo %in% rownames(tdm)) return(character(0))
  colnames(tdm)[tdm[termo, ] > 0]
}
head(busca_booleana("praia", tdm))
head(busca_booleana("bairro", tdm))

[1] "gonzaga-santos-p7"      "boqueirao-santos-p2"    "boqueirao-santos-p4"   
[4] "ponta-da-praia-p1"      "ponta-da-praia-p2"      "campo-grande-santos-p1"

[1] "gonzaga-santos-p1" "gonzaga-santos-p2" "gonzaga-santos-p3"
[4] "gonzaga-santos-p4" "gonzaga-santos-p5" "gonzaga-santos-p6"

In [ ]:
#TF-IDF

N <- ncol(tdm) #conta quantas colunas tem (nesse caso, quantos docs)
df <- rowSums(tdm > 0) # em quantos docs cada termo aparece
idf <- log(N / df) # quantos docs tem/ quantos docs a palavra aparece -> quanto  mais perto de zero, mais comum/menos informativa
# a palavra é. Quanto mais alto(longe de zero), mais rara/mais informativa é. #Log serve só pra suavizar
tfidf <- tdm * idf  # recicla idf por linha
round(tfidf[c("praia","bairro","forte","cidade"), ], 2)


,gonzaga-santos-p1,gonzaga-santos-p2,gonzaga-santos-p3,gonzaga-santos-p4,gonzaga-santos-p5,gonzaga-santos-p6,gonzaga-santos-p7,boqueirao-santos-p1,boqueirao-santos-p2,boqueirao-santos-p3,⋯,praia-grande-sao-paulo-p56,praia-grande-sao-paulo-p57,praia-grande-sao-paulo-p58,praia-grande-sao-paulo-p59,praia-grande-sao-paulo-p60,praia-grande-sao-paulo-p61,praia-grande-sao-paulo-p62,praia-grande-sao-paulo-p63,praia-grande-sao-paulo-p64,praia-grande-sao-paulo-p65
praia,0.00,0.00,0.00,0.00,0.00,0.00,1.24,0.00,1.24,0.00,⋯,0.00,0.00,1.24,0,1.24,0.00,1.24,1.24,1.24,0
bairro,1.71,1.71,3.43,1.71,1.71,1.71,1.71,5.14,0.00,1.71,⋯,0.00,0.00,0.00,0,0.00,0.00,1.71,1.71,0.00,0
forte,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0
cidade,1.32,0.00,0.00,1.32,2.63,0.00,0.00,1.32,0.00,0.00,⋯,1.32,1.32,0.00,0,1.32,1.32,1.32,0.00,1.32,0


## **Análise de similaridade de cosseno**

In [ ]:
N         <- ncol(tdm)
df        <- rowSums(tdm > 0)
idf_tfidf <- log(N / df)
tfidf     <- tdm * idf_tfidf

similaridade_cosseno <- function(a, b) {
  na <- sqrt(sum(a^2)); nb <- sqrt(sum(b^2))
  if (na == 0 || nb == 0) return(0)
  sum(a * b) / (na * nb)
}

vetorizar_consulta_tfidf <- function(consulta, vocab, idf) {
  tk <- tokenizar(consulta)
  contagem <- as.integer(table(factor(tk, levels = vocab)))
  contagem * idf
}

busca_vetorial <- function(consulta, vocab, idf, tfidf, docs, top_n = 5) {
  vq <- vetorizar_consulta_tfidf(consulta, vocab, idf)
  sims <- apply(tfidf, 2, function(col) similaridade_cosseno(vq, col))
  ranking <- sort(sims, decreasing = TRUE)[1:top_n]
  data.frame(
    documento = names(ranking),
    score = round(as.numeric(ranking), 3),
    texto = docs[names(ranking)],
    row.names = NULL
  )
}

busca_vetorial("praia bairro", vocab, idf_tfidf, tfidf, docs, top_n = 5)

documento,score,texto
<chr>,<dbl>,<chr>
ponta-da-praia-p1,0.281,"Ponta da Praia é um bairro da cidade paulista de Santos, no Brasil."
aparecida-santos-p1,0.225,"Aparecida é um bairro localizado na cidade de Santos,é o segundo bairro mais populoso do município com 36.440 habitantes."
jose-menino-p1,0.187,"José Menino é um bairro da cidade de Santos, São Paulo."
boqueirao-santos-p1,0.169,"Boqueirão é um bairro nobre localizado na zona leste da cidade paulista de Santos. Bairro autossuficiente e com grande infraestrutura, o Boqueirão conta com hospitais, universidades e escolas, supermercados, farmácias, agências bancárias, bares, ruas arborizadas e casas bem cuidadas. É um bairro tipicamente residencial e comercial."
boqueirao-santos-p6,0.159,"Hoje os imóveis do bairro estão supervalorizados em razão do ""boom"" imobiliário pelo qual passa a Baixada Santista. Embora o bairro ainda seja habitado pelas famílias tradicionais da cidade, o número de pessoas de classe alta oriundas de São Paulo que adquirem imóveis no bairro vem crescendo nos últimos anos."


# **Aula 3**

## **Normalização e Stopwords**

In [ ]:
limpar <- function(x) {
  x <- tolower(x)
  x <- iconv(x, to = "ASCII//TRANSLIT")
  x <- gsub("[^a-z0-9 ]", " ", x)
  x <- gsub("\\s+", " ", x)
  trimws(x)
}

stopwords <- c("da","do","de","o","a","os","as","e","em","um","uma",
               "uns","umas","no","na","por","como","para","mas")

tok      <- function(x) unlist(strsplit(limpar(x), " "))
sem_stop <- function(x) { t <- tok(x); t[!t %in% stopwords] }

sem_stop(docs[["gonzaga-santos-p1"]])

[1] "gonzaga"       "bairro"        "nobre"         "cidade"       
 [5] "santos"        "praca"         "independencia" "situada"      
 [9] "gonzaga"       "ponto"         "referencia"    "bairro"       
[13] "local"         "onde"          "sao"           "feitas"       
[17] "manifestacoes" "apresentacoes" "comemoracoes"

## **SnowBall**

In [ ]:
install.packages("SnowballC")
library(SnowballC)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
wordStem(c("cidade", "cidades", "cidadania", "cidadao"), language="portuguese")

[1] "cidad"   "cidad"   "cidadan" "cidada"

In [ ]:
docs

gonzaga-santos-p1 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       "O Gonzaga é um bairro nobre da cidade de Santos. A Praça da Independência, situada no Gonzaga é um ponto de referência do bairro, local onde são feitas manifestações, apresentações e comemorações." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            gonzaga-santos-p2 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                "O bairro do Gonzaga, fica delimitado pela orla da praia, os canais 2 e 3, e pela Avenida General Francisco Glycerio. Seus principais atrativos são três shoppings, sorveterias, a Praça da Independência, cine

## **BPE**

In [ ]:
frq <- c(praia = 10, bairro = 9, cidade =7)

simbolos <- lapply(names(frq), function(p) strsplit(p, "")[[1]])
names(simbolos) <- names(frq)
simbolos[["praia"]]

[1] "p" "r" "a" "i" "a"

In [ ]:
contar_pares <- function(simb, frq) {
  pares <- integer(0)
  for (p in names(simb)) {
    s <- simb[[p]]
    for (i in seq_len(length(s) - 1)) {
      k <- paste0(s[i], s[i + 1])
      pares[k] <- ifelse(is.na(pares[k]), 0, pares[k]) + frq[[p]]
    }
  }
  sort(pares, decreasing = TRUE)
}
head(contar_pares(simbolos, frq), 4)


ai pr ra ia 
19 10 10 10

In [ ]:
palavras <- c(praia = "p r a i a", bairro = "b a i r r o", cidade = "c i d a d e")

gsub("a i", "ai", palavras)

praia        bairro        cidade 
   "p r ai a"  "b ai r r o" "c i d a d e"

## **INDICE e BUSCA AND**

In [ ]:
postings <- list()
for (d in names(docs)) {
  for (termo in unique(sem_stop(docs[[d]]))) {
    postings[[termo]] <- c(postings[[termo]], d)
  }
}
length(postings)
head(postings[["praia"]])

busca_and <- function(consulta) {
  termos <- sem_stop(consulta)
  Reduce(intersect, postings[termos])
}
busca_and(c("praia", "bairro"))

[1] 3305

[1] "gonzaga-santos-p2"   "gonzaga-santos-p3"   "gonzaga-santos-p7"  
[4] "boqueirao-santos-p2" "boqueirao-santos-p4" "ponta-da-praia-p1"

[1] "gonzaga-santos-p2"          "gonzaga-santos-p3"         
 [3] "gonzaga-santos-p7"          "ponta-da-praia-p1"         
 [5] "ponta-da-praia-p2"          "campo-grande-santos-p1"    
 [7] "praia-do-tombo-p1"          "sao-vicente-sao-paulo-p3"  
 [9] "praia-grande-sao-paulo-p8"  "praia-grande-sao-paulo-p20"
[11] "praia-grande-sao-paulo-p44" "praia-grande-sao-paulo-p49"
[13] "praia-grande-sao-paulo-p62" "praia-grande-sao-paulo-p63"

# **Aula 4**

In [ ]:
# Recalcula a matriz com a tokenização limpa
tokens_limpos <- lapply(docs, sem_stop)
vocab_limpo   <- sort(unique(unlist(tokens_limpos)))
length(vocab_limpo)

tf <- sapply(tokens_limpos, function(tk) {
  as.integer(table(factor(tk, levels = vocab_limpo)))
})
rownames(tf) <- vocab_limpo

N     <- ncol(tf)             # 194
df    <- rowSums(tf > 0)
dl    <- colSums(tf)
avgdl <- mean(dl)

idf_classico <- log(N / df)                         # para o cosseno
idf_bm25     <- log((N - df + 0.5) / (df + 0.5) + 1) # para o BM25

tfidf_cos <- tf * idf_classico

[1] 3305

In [ ]:
k1 <- 1.2; b <- 0.75

bm25_doc <- function(termos, d) {
  s <- 0
  for (t in termos) {
    if (!(t %in% vocab_limpo)) next
    f <- tf[t, d]
    K <- k1 * (1 - b + b * dl[d] / avgdl)
    s <- s + idf_bm25[t] * (f * (k1 + 1)) / (f + K)
  }
  s
}

In [ ]:
nec <- read.csv("necessidades.csv", stringsAsFactors = FALSE)
nec[, c("consulta", "texto_consulta")]

vetorizar_consulta_cos <- function(consulta) {
  tk <- sem_stop(consulta)
  contagem <- as.integer(table(factor(tk, levels = vocab_limpo)))
  contagem * idf_classico
}

rankings <- list()
for (i in seq_len(nrow(nec))) {
  qid    <- nec$consulta[i]
  qtxt   <- nec$texto_consulta[i]
  termos <- sem_stop(qtxt)

  # BM25
  s_bm25 <- sapply(colnames(tf), function(d) unname(bm25_doc(termos, d)))
  r_bm25 <- names(sort(s_bm25, decreasing = TRUE))

  # Cosseno (mesmo vocab, mesmo pré-processamento)
  vq    <- vetorizar_consulta_cos(qtxt)
  s_cos <- apply(tfidf_cos, 2, function(col) similaridade_cosseno(vq, col))
  r_cos <- names(sort(s_cos, decreasing = TRUE))

  rankings[[qid]] <- list(bm25 = r_bm25, cos = r_cos)
}

# Confira antes de salvar:
head(rankings[["q01"]]$bm25, 5)   # devem aparecer gonzaga-santos-p* no topo
head(rankings[["q01"]]$cos, 5)

# Salvar
linhas <- do.call(rbind, lapply(names(rankings), function(qid) {
  data.frame(
    consulta = qid,
    posicao  = seq_along(rankings[[qid]]$bm25),
    doc_bm25 = rankings[[qid]]$bm25,
    doc_cos  = rankings[[qid]]$cos
  )
}))
write.csv(linhas, "rankings.csv", row.names = FALSE)
head(linhas, 10)

consulta,texto_consulta
<chr>,<chr>
q01,lugar calmo no gonzaga
q02,cubatao recuperacao ambiental poluicao atual
q03,monumentos historicos fundacao sao vicente
q04,praia do tombo surf ondas iniciante
q05,ciclovias orla praia grande continua segura


[1] "gonzaga-santos-p1" "gonzaga-santos-p3" "jose-menino-p4"   
[4] "gonzaga-santos-p7" "gonzaga-santos-p2"

[1] "gonzaga-santos-p1" "gonzaga-santos-p3" "jose-menino-p4"   
[4] "gonzaga-santos-p7" "gonzaga-santos-p2"

,consulta,posicao,doc_bm25,doc_cos
,<chr>,<int>,<chr>,<chr>
1,q01,1,gonzaga-santos-p1,gonzaga-santos-p1
2,q01,2,gonzaga-santos-p3,gonzaga-santos-p3
3,q01,3,jose-menino-p4,jose-menino-p4
4,q01,4,gonzaga-santos-p7,gonzaga-santos-p7
5,q01,5,gonzaga-santos-p2,gonzaga-santos-p2
6,q01,6,gonzaga-santos-p5,gonzaga-santos-p5
7,q01,7,boqueirao-santos-p2,guaruja-p7
8,q01,8,guaruja-p7,boqueirao-santos-p2
9,q01,9,gonzaga-santos-p6,gonzaga-santos-p6


# **Aula 5.5**

## módulo 11

In [ ]:
q <- read.csv("qrels_.csv")                          # o gabarito exportado pelo julgar.html
head(q)                                             # as primeiras linhas: consulta, documento, grau, ...

Warning message in file(file, "rt"):
“cannot open file 'qrels_.csv': No such file or directory”


ERROR: Error in file(file, "rt"): cannot open the connection


In [ ]:
linhas_q01 <- q[q$consulta == "q01", ]
grau <- setNames(linhas_q01$grau, linhas_q01$documento)
grau

LIMIAR <- 2
#Escolho limiar ≥ 2 porque as necessidades do projeto pedem respostas diretas (ex.: "as ondas do Tombo servem para iniciante?"),
#e um documento que só diz "fica ao lado das Astúrias" é contexto, não resposta — contá-lo como relevante infla o recall com documentos que não resolvem a pergunta.
relevantes <- names(grau)[grau >= LIMIAR]
relevantes
R <- length(relevantes)
R                 # 2  (gonzaga-santos-p1, gonzaga-santos-p7)

In [ ]:
ranking_bm25 <- rankings[["q01"]]$bm25
ranking_cos  <- rankings[["q01"]]$cos

rel_bm25 <- as.integer(ranking_bm25 %in% relevantes)
rel_cos  <- as.integer(ranking_cos  %in% relevantes)

head(ranking_bm25, 10)
rel_bm25[1:10]
rel_cos[1:10]

R
sum(rel_bm25[1:3])

In [ ]:
dcg <- function(v) {
  sum(v / log2(seq_along(v) + 1))
}

In [ ]:
R        <- length(relevantes)
precisao <- cumsum(rel_bm25) / seq_along(rel_bm25)
ap       <- sum(precisao[rel_bm25 == 1]) / R
mrr      <- 1 / which(rel_bm25 == 1)[1]
ndcg     <- dcg(rel_bm25) / dcg(sort(rel_bm25, decreasing = TRUE))

# Para o nDCG graduado (tratando NA como 0):
g        <- grau[ranking_bm25]
g[is.na(g)] <- 0
ndcg_g   <- dcg(g) / dcg(sort(g, decreasing = TRUE))

In [ ]:
c(P_at_3   = sum(rel_bm25[1:3]) / 3,
  AP       = ap,
  MRR      = mrr,
  nDCG_bin = ndcg,
  nDCG_grad = ndcg_g)

In [ ]:
precisao_cos <- cumsum(rel_cos) / seq_along(rel_cos)
ap_cos   <- sum(precisao_cos[rel_cos == 1]) / R
mrr_cos  <- 1 / which(rel_cos == 1)[1]
ndcg_cos <- dcg(rel_cos) / dcg(sort(rel_cos, decreasing = TRUE))

g_cos <- grau[ranking_cos]
g_cos[is.na(g_cos)] <- 0
ndcg_g_cos <- dcg(g_cos) / dcg(sort(g_cos, decreasing = TRUE))

c(P_at_3   = sum(rel_cos[1:3]) / 3,
  AP       = ap_cos,
  MRR      = mrr_cos,
  nDCG_bin = ndcg_cos,
  nDCG_grad = ndcg_g_cos)

## Módulo 12

In [ ]:
# Função que calcula as cinco métricas para UM ranking + UM grau
metricas <- function(rel, ranking, grau) {
  R        <- sum(grau >= 2)          # total de relevantes na consulta
  precisao <- cumsum(rel) / seq_along(rel)
  ap       <- sum(precisao[rel == 1]) / R
  mrr      <- if (any(rel == 1)) 1 / which(rel == 1)[1] else 0
  ndcg_bin <- dcg(rel) / dcg(sort(rel, decreasing = TRUE))

  g <- grau[ranking]
  g[is.na(g)] <- 0
  ndcg_g <- dcg(g) / dcg(sort(g, decreasing = TRUE))

  c(P_at_3 = sum(rel[1:3]) / 3,
    AP = ap, MRR = mrr,
    nDCG_bin = ndcg_bin, nDCG_grad = ndcg_g)
}

# Loop nas 5 consultas
res <- list()
for (qid in sprintf("q%02d", 1:5)) {
  linhas <- q[q$consulta == qid, ]
  if (nrow(linhas) == 0) next

  grau_q <- setNames(linhas$grau, linhas$documento)
  rel_b  <- as.integer(rankings[[qid]]$bm25 %in% names(grau_q)[grau_q >= 2])
  rel_c  <- as.integer(rankings[[qid]]$cos  %in% names(grau_q)[grau_q >= 2])

  res[[qid]] <- list(
    R     = sum(grau_q >= 2),
    bm25  = metricas(rel_b, rankings[[qid]]$bm25, grau_q),
    cos   = metricas(rel_c, rankings[[qid]]$cos,  grau_q)
  )
}

# Tabela por consulta
for (qid in names(res)) {
  cat("\n--- ", qid, " (R = ", res[[qid]]$R, ") ---\n", sep = "")
  print(round(rbind(bm25 = res[[qid]]$bm25, cos = res[[qid]]$cos), 3))
}

# MAP de cada sistema
ap_bm25 <- sapply(res, function(x) x$bm25["AP"])
ap_cos  <- sapply(res, function(x) x$cos["AP"])
cat("\nMAP BM25: ", round(mean(ap_bm25), 3), "\n")
cat("MAP cos:  ", round(mean(ap_cos),  3), "\n")

In [ ]:
qid <- "q05"
linhas <- q[q$consulta == qid, ]
grau_q <- setNames(linhas$grau, linhas$documento)
relev_q <- names(grau_q)[grau_q >= 2]

relev_q                                    # o único relevante
head(rankings[[qid]]$bm25, 20)             # o BM25 achou ele?
head(rankings[[qid]]$cos,  20)             # o cosseno achou ele?

rel_b <- as.integer(rankings[[qid]]$bm25 %in% relev_q)
rel_c <- as.integer(rankings[[qid]]$cos  %in% relev_q)
which(rel_b == 1)                          # em que posição o BM25 pôs
which(rel_c == 1)                          # em que posição o cosseno pôs

# Aula 6

In [ ]:
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor06.R")  # o que veio das Aulas 00 a 5,5
docs <- docs_aula()          # os 8 documentos do curso
cfg  <- cfg_aula()           # as decisoes canonicas (agora com limiar = 2)
ix   <- montar(docs, cfg)    # tudo que as aulas anteriores calcularam
estado()                     # confira: MOTOR_VERSAO "motor06 ..."

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-10-23 15:19 · R version 4.6.1 (2026-06-24) · motor06 v2 2026-10-08 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | manter |
| `cfg$stopwords` | character | 0 |  |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$limiar` | numeric | 1 | 2 |
| `docs` | character, nomeado | 8 | d1 recuperacao de informacao o..., d2 o modelo de espaco vetorial..., d3 bm25 e um modelo probabilis..., d4 aprendizado estatistico fun..., d5 o indice invertido acelera ..., d6 embeddings capturam a seman..., ... |
| `g` | numeric, nomeado | 8 | d1 1, d2 2, d3 2, d4 0, d5 0, d6 1, ... |
| `ix$tokens` | list | 8 | elementos: d1, d2, d3, d4, d5, d6, d7, d8 |
| `ix$vocab` | character | 45 | a, acelera, aprendizado, avaliacao, bm25, busca, ... |
| `ix$tf` | matrix | 45 × 8 | linhas:

In [ ]:
base <- ranking_cosseno("modelo de recuperacao", ix, cfg)  # o ranking da Aula 02
round(base, 3)                                             # escores com nomes
grau <- gabarito_aula()                                    # os graus da Aula 05
round(metricas(base, grau, 2), 3)                          # as cinco da Aula 5,5, limiar 2

# Saída esperada:
#    d1    d3    d4    d2    d6    d8    d5    d7
# 0.254 0.233 0.215 0.208 0.025 0.023 0.000 0.000
#    P_at_k        AP        RR  nDCG_bin nDCG_grad
#     0.333     0.500     0.500     0.651     0.837

# Note que o usuário olha os 4 primeiros resultados (d1, d3, d4, d2):
# * Marca d2 e d3 como relevantes (grau 2), formando o conjunto Dr.
# * Marca d4 como não relevante (grau 0), formando o conjunto Dnr.
# * d1 (grau 1) fica sem marca pois está abaixo do limiar 2.

# Lembrete importante: As marcas do usuário são colocadas sobre documentos, e não sobre palavras diretamente.
# O motor de busca é quem vai deduzir quais palavras importam.

# ---

# ### Checkpoint 1
# Se o usuário olhasse apenas o top-3 do ranking (d1, d3, d4) e marcasse segundo o gabarito (limiar 2),
# quem iria para Dr e Dnr? Que documento relevante ele deixaria de ver?

d1    d3    d4    d2    d6    d8    d5    d7 
0.254 0.233 0.215 0.208 0.025 0.023 0.000 0.000

P_at_k        AP        RR  nDCG_bin nDCG_grad 
    0.333     0.500     0.500     0.651     0.837

In [ ]:
q0 <- vetor_consulta(tokenizar("modelo de recuperacao"), ix$vocab, ix$idf_tfidf)  # 45 pesos, a regua do corpus
length(q0)              # quantas coordenadas (45)
round(q0[q0 > 0], 3)    # so as nao nulas

# Saída esperada:
# [1] 45
#          de      modelo recuperacao
#       0.470       1.386       1.386

# Para mover esse vetor em direção aos documentos relevantes Dr, calculamos o centroide
# (a média das colunas das matrizes de pesos dos documentos em Dr):
# cr = 1/|Dr| * sum_{d em Dr} d

# Em R, usamos a função rowMeans() para obter a média de cada linha (termo a termo):

round(ix$w[c("de", "modelo", "espaco"), c("d2", "d3")], 3)  # tres linhas, duas colunas
cr <- rowMeans(ix$w[, c("d2", "d3")])                       # o centroide: media das colunas, termo a termo
round(cr[c("de", "modelo", "espaco")], 3)                   # as mesmas tres linhas

# Saída esperada:
#           d2    d3
# de     0.470 0.940
# modelo 1.386 1.386
# espaco 2.079 0.000
#
#     de modelo espaco
#  0.705  1.386  1.040

# Observe como o termo espaco (presente em d2, mas ausente da consulta original q0) passa a ter peso no centroide cr (1.040)!

# Atenção prática no R: Quando selecionar um único documento em uma matriz, utilize sempre
# drop = FALSE (ex: ix$w[, "d4", drop = FALSE]) para evitar que a matriz perca a estrutura
# bidimensional e cause erro no rowMeans().

# ---

# ### Checkpoint 2
# Qual é o centroide de três documentos (d1, d3 e d4) para o termo recuperacao?
# (Considere que d1 e d4 têm peso 1.386, enquanto d3 tem peso 0 para esse termo).
# Calcule a média simples à mão ou no R.

[1] 45

de      modelo recuperacao 
      0.470       1.386       1.386

,d2,d3
de,0.470,0.940
modelo,1.386,1.386
espaco,2.079,0.000


de modelo espaco 
 0.705  1.386  1.040

In [ ]:
rocchio <- function(q, Dr, Dnr = character(0), ix, a = 1, b = 0.75, g = 0.15) {  # q: vetor; Dr, Dnr: nomes
  cr  <- rowMeans(ix$w[, Dr, drop = FALSE])                                # centroide dos relevantes
  cnr <- if (length(Dnr)) rowMeans(ix$w[, Dnr, drop = FALSE]) else 0       # dos nao relevantes; sem nenhum, 0
  a * q + b * cr - g * cnr                                                 # a formula, termo a termo
}                                                                          # fim da funcao

qm <- rocchio(q0, Dr = c("d2", "d3"), Dnr = "d4", ix = ix)  # o gabarito como marcas do usuario
sum(qm != 0)                                                # quantos termos tem a consulta agora
round(head(sort(qm, decreasing = TRUE), 8), 3)              # os de maior peso
round(qm[qm < 0], 3)                                        # os de peso negativo

# Saída esperada:
# [1] 21
#         modelo    recuperacao             de           bm25         como
#          2.426          1.178          0.999          0.780        0.780
#         espaco probabilistico   ranqueamento
#          0.780          0.780          0.780
#           a aprendizado estatistico  fundamenta     moderna
#      -0.104      -0.312      -0.312      -0.312      -0.312

# A consulta passou de 3 para 21 termos (Expansão de Consulta) e 5 termos associados a d4 receberam
# pesos negativos por causa do fator gama!

# ---

# ### Checkpoint 3
# Calcule à mão o peso do termo recuperacao no vetor reformulado qm:
# * Peso em q0 = 1.386
# * Em Dr = {d2, d3}: d2 = 0, d3 = 0
# * Em Dnr = {d4}: d4 = 1.386
#
# Utilizando alfa = 1, beta = 0.75 e gama = 0.15, quanto dá 1 * 1.386 + 0.75 * 0 - 0.15 * 1.386?

[1] 21

modelo    recuperacao             de           bm25           como 
         2.426          1.178          0.999          0.780          0.780 
        espaco probabilistico   ranqueamento 
         0.780          0.780          0.780

a aprendizado estatistico  fundamenta     moderna 
     -0.104      -0.312      -0.312      -0.312      -0.312

In [ ]:
# Função para ranquear um vetor qualquer contra o índice
ranking_vetor <- function(q, ix) {
  sort(apply(ix$w, 2, function(d) cosseno(q, d)), decreasing = TRUE)
}

# Teste com a consulta original (deve dar o ranking base)
round(ranking_vetor(q0, ix), 3)

# Agora calculamos o novo ranking com a consulta do Rocchio
novo <- ranking_vetor(qm, ix)
round(novo, 3)

# Comparação das métricas antes e depois
round(rbind(base = metricas(base, grau, 2), rocchio = metricas(novo, grau, 2)), 3)

# Saída esperada das métricas:
#         P_at_k  AP  RR nDCG_bin nDCG_grad
# base     0.333 0.5 0.5    0.651     0.837
# rocchio  0.667 1.0 1.0    1.000     1.000

# O AP saltou de 0.500 para 1.000! Os documentos relevantes (d3 e d2) foram para a 1ª e 2ª posições,
# enquanto o não relevante d4 caiu para a última posição, com cosseno negativo (-0.060) devido aos
# pesos negativos de qm.

# ---

# ### Checkpoint 4
# Se usarmos a função pmax(qm, 0) para zerar todos os pesos negativos de qm, o ranking fica:
# d3, d2, d1, d4, d6, d5, d8, d7
#
# Com esse ajuste, o AP continua em 1.000, mas o nDCG graduado cai de 1.000 para 0.990.
# Por que o nDCG graduado cai, enquanto o AP permanece em 1.000?

d1    d3    d4    d2    d6    d8    d5    d7 
0.254 0.233 0.215 0.208 0.025 0.023 0.000 0.000

d3     d2     d1     d6     d8     d5     d7     d4 
 0.650  0.643  0.140  0.055  0.045  0.041 -0.007 -0.060

,P_at_k,AP,RR,nDCG_bin,nDCG_grad
base,0.333,0.5,0.5,0.651,0.837
rocchio,0.667,1.0,1.0,1.000,1.000


In [ ]:
# Remover os documentos já julgados (d2, d3, d4)
fora  <- c("d2", "d3", "d4")
g_res <- grau[!names(grau) %in% fora]
b_res <- base[!names(base) %in% fora]
n_res <- novo[!names(novo) %in% fora]

# Comparação do ranking base vs Rocchio com limiar 1 na coleção residual
round(rbind(base = metricas(b_res, g_res, 1), rocchio = metricas(n_res, g_res, 1)), 3)

# Saída esperada:
#         P_at_k AP RR nDCG_bin nDCG_grad
# base     0.667  1  1        1         1
# rocchio  0.667  1  1        1         1

# Resultado: Empate técnico em 1,000 para todas as métricas!
# Nos documentos não vistos (d1, d6, d8, d5, d7), a ordem relativa mantida tanto pelo base quanto pelo Rocchio foi exatamente a mesma (d1 em 1º e d6 em 2º). O ganho aparente no Módulo 4 vinha apenas da reordenação dos documentos já conhecidos.

# ---

# ### Checkpoint 5
# Considere outro cenário: o usuário olhou o top-3 e marcou apenas d3 (relevante) e d4 (não relevante).
# O documento d2 (relevante grau 2) não foi visto nem marcado pelo usuário, permanecendo na coleção residual.
#
# Se rodarmos o Rocchio com Dr = {d3} e Dnr = {d4}, o que você prevê que acontecerá com o AP na coleção residual (com limiar 2) em relação ao ranking base? Ele subirá ou empatará?

,P_at_k,AP,RR,nDCG_bin,nDCG_grad
base,0.667,1,1,1,1
rocchio,0.667,1,1,1,1


In [ ]:
top <- names(base)[1:2]
q_prf <- rocchio(q0, Dr = top, ix = ix)
round(head(sort(q_prf, decreasing = TRUE), 6), 3)
r_prf <- ranking_vetor(q_prf, ix)
round(r_prf, 3)
round(metricas(r_prf, grau, 2), 3)

modelo recuperacao          de        bm25  informacao      ordena 
      1.906       1.906       0.999       0.780       0.780       0.780

d3    d1    d2    d4    d6    d8    d7    d5 
0.648 0.575 0.175 0.161 0.062 0.047 0.037 0.010

P_at_k        AP        RR  nDCG_bin nDCG_grad 
    0.667     0.833     1.000     0.920     0.958

# Parte D - 6

In [ ]:
cfg_linhas <- c(
"cfg <- list(",
"  limiar = 2,",
"  juizes = NULL,",
"  prf_k = 0,",
"  rocchio = c(a = 1, b = 0.75, g = 0.15)",
")"
)
writeLines(cfg_linhas, "config.R")
source("config.R")

In [ ]:
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor06.R")
cp   <- read.csv("corpus.csv", stringsAsFactors = FALSE, fileEncoding = "UTF-8")
docs <- setNames(cp$texto, cp$id)
ix   <- montar(docs, cfg)
estado()

Warning message in file(file, "rt", encoding = fileEncoding):
“cannot open file 'corpus.csv': No such file or directory”


ERROR: Error in file(file, "rt", encoding = fileEncoding): cannot open the connection


In [ ]:
rocchio <- function(q, Dr, Dnr = character(0), ix, a = 1, b = 0.75, g = 0.15) {
  cr  <- rowMeans(ix$w[, Dr, drop = FALSE])
  cnr <- if (length(Dnr)) rowMeans(ix$w[, Dnr, drop = FALSE]) else 0
  a * q + b * cr - g * cnr
}

ranking_vetor <- function(q, ix) {
  sort(apply(ix$w, 2, function(d) cosseno(q, d)), decreasing = TRUE)
}

In [ ]:
residual <- function(rk, g, fora, limiar) {
metricas(rk[!names(rk) %in% fora], g[!names(g) %in% fora], limiar)
}

In [ ]:
ix_aula <- montar(docs_aula(), cfg_aula())
q_aula  <- vetor_consulta(tokenizar("modelo de recuperacao"), ix_aula$vocab, ix_aula$idf_tfidf)
novo_aula <- ranking_vetor(rocchio(q_aula, c("d2", "d3"), "d4", ix_aula), ix_aula)
head(novo_aula, 8)

In [ ]:
prf <- function(q0, k, ix, p = c(a = 1, b = 0.75, g = 0.15)) {
top <- names(ranking_vetor(q0, ix))[1:k]
ranking_vetor(rocchio(q0, top, ix = ix, a = p[["a"]], b = p[["b"]], g = p[["g"]]), ix)
}

In [ ]:
ap_q <- function(txt, grau, k, ix, cfg) {
q0 <- vetor_consulta(preparar(txt, cfg), ix$vocab, ix$idf_tfidf)
rk <- if (k == 0) ranking_vetor(q0, ix) else prf(q0, k, ix)
metricas(rk, grau, cfg$limiar)[["AP"]]
}

In [ ]:
arqs <- list.files(pattern = "qrels_", fixed = TRUE)
if (length(arqs) == 0) {
  arqs <- list.files(pattern = "qrels")
}

qrels <- ler_qrels(arqs, juiz = cfg$juizes)
nec   <- read.csv("necessidades.csv", stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
consultas <- setNames(nec$texto_consulta, nec$consulta)

R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))
usar <- names(qrels)[R_q > 0]

In [ ]:
tab_dev <- sapply(usar, function(q) {
a <- function(k) ap_q(consultas[[q]], qrels[[q]], k, ix, cfg)
c(base = a(0), k2 = a(2), k3 = a(3), k5 = a(5))
})

In [ ]:
round(tab_dev, 3)
round(rowMeans(tab_dev), 3)

In [ ]:
cfg_linhas <- c(
"cfg <- list(",
"  limiar = 2,",
"  juizes = NULL,",
"  prf_k = 3,",
"  rocchio = c(a = 1, b = 0.75, g = 0.15)",
")"
)
writeLines(cfg_linhas, "config.R")
source("config.R")

In [ ]:
ranking_motor <- function(txt, ix, cfg) {
q0 <- vetor_consulta(preparar(txt, cfg), ix$vocab, ix$idf_tfidf)
if (cfg$prf_k == 0) {
ranking_vetor(q0, ix)
} else {
prf(q0, cfg$prf_k, ix, cfg$rocchio)
}
}

In [ ]:
round(head(ranking_motor(consultas[[usar[1]]], ix, cfg), 5), 3)

# **AULA 7**

In [ ]:
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor07.R")  # o que veio das Aulas 00 a 06
docs <- docs_aula()          # os 8 documentos do curso
cfg  <- cfg_aula()           # as decisoes canonicas (agora com prf_k e rocchio)
ix   <- montar(docs, cfg)    # tudo que as aulas anteriores calcularam
estado()                     # confira: MOTOR_VERSAO "motor07 ..."

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-10-23 14:16 · R version 4.6.1 (2026-06-24) · motor07 v1 2026-10-15 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | manter |
| `cfg$stopwords` | character | 0 |  |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$limiar` | numeric | 1 | 2 |
| `cfg$prf_k` | numeric | 1 | 0 |
| `cfg$rocchio` | numeric, nomeado | 3 | a 1, b 0.75, g 0.15 |
| `docs` | character, nomeado | 8 | d1 recuperacao de informacao o..., d2 o modelo de espaco vetorial..., d3 bm25 e um modelo probabilis..., d4 aprendizado estatistico fun..., d5 o indice invertido acelera ..., d6 embeddings capturam a seman..., ... |
| `ix$tokens` | list | 8 | elementos: d1, d2, d3, d4, d5, d6, d7, d8 |
| `ix$vocab` | character | 45 | a, acelera, aprendizado, avaliacao, bm25, busca, ... |
| `ix$tf` | m

In [ ]:
round(ranking_cosseno("busca", ix, cfg), 3)          # quem tem a palavra "busca"?
round(ranking_cosseno("ranqueamento", ix, cfg), 3)   # e "ranqueamento"?
round(ranking_cosseno("ordenar documentos", ix, cfg), 3)

d5    d7    d1    d2    d3    d4    d6    d8 
0.270 0.265 0.000 0.000 0.000 0.000 0.000 0.000

d3    d1    d2    d4    d5    d6    d7    d8 
0.413 0.000 0.000 0.000 0.000 0.000 0.000 0.000

d1    d6    d2    d5    d3    d4    d7    d8 
0.166 0.157 0.136 0.135 0.000 0.000 0.000 0.000

In [ ]:
sum(ix$w[, "d4"] == 0)                    # quantas coordenadas de d1 nao sao zero
sum(ix$w != 0) / length(ix$w)             # a fracao nao nula da matriz inteira

[1] 39

[1] 0.1722222

In [ ]:
lsa <- function(ix, k) {                                          # k: quantas dimensoes guardar
  s <- svd(ix$w)                                                  # W = U Sigma V'
  U <- s$u[, 1:k, drop = FALSE]; rownames(U) <- ix$vocab          # termos x k
  D <- s$v[, 1:k, drop = FALSE] %*% diag(s$d[1:k], nrow = k)      # documentos x k: V_k Sigma_k
  rownames(D) <- colnames(ix$w)                                   # d1 ... d8
  list(U = U, S = s$d[1:k], D = D)                                # o modelo
}

In [ ]:
m2 <- lsa(ix, 2)       # LSA com k = 2
round(m2$D, 3)         # os 8 documentos em duas coordenadas

d1,-0.862,-0.253
d2,-2.417,2.910
d3,-1.340,3.089
d4,-0.583,-0.397
d5,-3.361,-0.685
d6,-0.945,0.428
d7,-3.207,-2.888
d8,-0.434,1.001


In [ ]:
q0 <- vetor_consulta(preparar("modelo de recuperacao", cfg), ix$vocab, ix$idf_tfidf)  # 45 pesos
qk <- t(m2$U) %*% q0                                       # (2 x 45) %*% (45): duas coordenadas
round(qk, 3)                                               # a consulta dobrada

-0.370
0.445


In [ ]:
ranking_lsa <- function(consulta, ix, cfg, modelo) {               # modelo: o que lsa() devolve
  q0 <- vetor_consulta(preparar(consulta, cfg), ix$vocab, ix$idf_tfidf)  # texto -> 45 pesos
  qk <- t(modelo$U) %*% q0                                         # dobra: k coordenadas
  s  <- apply(modelo$D, 1, function(d) cosseno(qk, d))             # um cosseno por LINHA (documento)
  sort(s, decreasing = TRUE)                                       # do maior ao menor
}

r2 <- ranking_lsa("modelo de recuperacao", ix, cfg, m2); round(r2, 3)    # o ranking LSA, k = 2
round(metricas(r2, gabarito_aula(), 2), 3)                               # as cinco, limiar 2

d2     d3     d8     d6     d5     d1     d4     d7 
 1.000  0.960  0.960  0.900  0.474  0.397  0.096 -0.039

P_at_k        AP        RR  nDCG_bin nDCG_grad 
    0.667     1.000     1.000     1.000     0.966

In [ ]:
round(ranking_lsa("busca", ix, cfg, m2), 3)          # no lexico, so d5 e d7
round(ranking_lsa("ranqueamento", ix, cfg, m2), 3)   # no lexico, so d3

d4     d7     d1     d5     d6     d2     d3     d8 
 0.999  0.982  0.967  0.942  0.566  0.149 -0.134 -0.134

d8     d3     d2     d6     d5     d1     d4     d7 
 0.999  0.999  0.949  0.715  0.170  0.086 -0.224 -0.353

In [ ]:
intersect(ix$tokens$d3, ix$tokens$d8)            # apenas "e" e "de"
round(cosseno(ix$w[, "d3"], ix$w[, "d8"]), 3)    # na TF-IDF: 0.058
round(cosseno(m2$D["d3", ], m2$D["d8", ]), 3)    # no LSA (k=2): 1.000

[1] "e"  "de"

[1] 0.058

[1] 1

In [ ]:
rrf <- function(..., c = 60) {                         # ...: quantos rankings vierem
  rks <- list(...)                                     # os rankings, numa lista
  ids <- unique(unlist(lapply(rks, names)))            # todos os documentos que aparecem
  s <- sapply(ids, function(d)                         # para cada documento...
    sum(sapply(rks, function(r) {                      # ...soma sobre os rankings
      p <- which(names(r) == d)                        # a posicao dele neste ranking
      if (length(p)) 1 / (c + p) else 0 })))           # ausente: nao soma nada
  sort(s, decreasing = TRUE)                           # do maior ao menor
}

In [ ]:
rb <- ranking_bm25("modelo de recuperacao", ix, cfg)   # lexico: d3 d1 d2 d4 d8 d6 d5 d7
r2 <- ranking_lsa("modelo de recuperacao", ix, cfg, m2)# denso:  d2 d3 d8 d6 d5 d1 d4 d7
rh <- rrf(rb, r2)                                      # funde os dois rankings
round(rh, 4)

d3     d2     d1     d8     d6     d4     d5     d7 
0.0325 0.0323 0.0313 0.0313 0.0308 0.0306 0.0303 0.0294

In [ ]:
rc <- ranking_cosseno("modelo de recuperacao", ix, cfg); g <- gabarito_aula()
round(rbind(cosseno = metricas(rc, g, 2), bm25 = metricas(rb, g, 2),
            lsa2 = metricas(r2, g, 2), rrf = metricas(rh, g, 2)), 3)

,P_at_k,AP,RR,nDCG_bin,nDCG_grad
cosseno,0.333,0.500,0.5,0.651,0.837
bm25,0.667,0.833,1.0,0.920,0.951
lsa2,0.667,1.000,1.0,1.000,0.966
rrf,0.667,1.000,1.0,1.000,0.990


# **AULA 7D**

In [7]:
REPO <- "https://raw.githubusercontent.com/Marques07/3mkt-pi3/main/"
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor07.R")

download.file(paste0(REPO, "Estrutura/config.R"), "config.R")
source("config.R")

cp   <- read.csv(paste0(REPO, "Estrutura/corpus.csv"),
                 stringsAsFactors = FALSE, fileEncoding = "UTF-8")
docs <- setNames(cp$texto, cp$id)

ix <- montar(docs, cfg)
estado()

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-10-23 20:56 · R version 4.6.1 (2026-06-24) · motor07 v1 2026-10-15 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `arvore` | data.frame | 17 × 6 | colunas: path, mode, type, sha, size, url |
| `cfg$minimo` | numeric | 1 | 40 |
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | translit |
| `cfg$stopwords` | character | 19 | da, do, de, o, a, os, ... |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$limiar` | numeric | 1 | 2 |
| `cfg$prf_k` | numeric | 1 | 0 |
| `cfg$rocchio` | numeric, nomeado | 3 | a 1, b 0.75, g 0.15 |
| `cfg$modelo` | character | 1 | bm25 |
| `cfg$lsa_k` | numeric | 1 | 20 |
| `cfg$rrf_c` | numeric | 1 | 60 |
| `cfg$grupo` | character | 1 | 3Mkt |
| `cp` | data.frame | 194 × 3 | colunas: id, titulo, texto |
| `docs` | character, nomeado | 194 | gonzaga-santos-p1 O Gonzaga é um

In [13]:
arqs <- "qrels.csv"
for (a in arqs) download.file(paste0(REPO, "Estrutura/", a), a)

qrels <- ler_qrels(arqs, juiz = cfg$juizes)

nec <- read.csv(paste0(REPO, "Estrutura/necessidades.csv"),
                stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
consultas <- setNames(nec$texto_consulta, nec$consulta)

R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))
usar <- names(qrels)[R_q > 0]

In [15]:
ler_qrels                      # mostra o código da função (veja o que faz com 'juiz')
head(read.csv("qrels.csv"))    # formato do arquivo
str(qrels)                     # o que foi carregado
length(qrels)
R_q
usar
all(usar %in% names(consultas))

function (arquivos, juiz = NULL) 
{
    ler <- function(f) read.csv(f, stringsAsFactors = FALSE, 
        fileEncoding = "UTF-8-BOM")
    q <- do.call(rbind, lapply(arquivos, ler))
    q$juiz <- trimws(q$juiz)
    q <- q[!grepl("_p2$", q$juiz), ]
    juizes <- unique(q$juiz)
    if (is.null(juiz)) {
        if (length(juizes) > 1) 
            stop("ha mais de um juiz: ", paste(juizes, collapse = ", "), 
                "\nDiga qual vale, ou a ordem de prioridade, em juiz = c(...)", 
                call. = FALSE)
        juiz <- juizes
    }
    q <- q[q$juiz %in% juiz, ]
    q <- q[order(match(q$juiz, juiz)), ]
    q <- q[!duplicated(paste(q$consulta, q$documento)), ]
    split(setNames(as.numeric(q$grau), q$documento), q$consulta)
}
<bytecode: 0x5bc428757788>

,consulta,documento,grau,juiz,timestamp,segundos
,<chr>,<chr>,<int>,<chr>,<chr>,<int>
1,q03,cubatao-p50,0,João Pedro,2026-09-17T20:26:45.622Z,122
2,q02,cubatao-p50,1,João Pedro,2026-09-17T20:27:40.461Z,55
3,q03,guaruja-p14,0,João Pedro,2026-09-17T20:28:09.652Z,29
4,q05,sao-vicente-sao-paulo-p21,0,João Pedro,2026-09-17T20:28:40.971Z,31
5,q03,sao-vicente-sao-paulo-p10,1,João Pedro,2026-09-17T20:28:57.411Z,16
6,q04,guaruja-p20,1,João Pedro,2026-09-17T20:29:58.571Z,61


List of 5
 $ q01: Named num [1:10] 1 0 1 2 1 0 0 0 0 2
  ..- attr(*, "names")= chr [1:10] "gonzaga-santos-p3" "jose-menino-p4" "gonzaga-santos-p5" "gonzaga-santos-p1" ...
 $ q02: Named num [1:11] 1 2 1 0 0 2 1 0 0 0 ...
  ..- attr(*, "names")= chr [1:11] "cubatao-p50" "cubatao-p23" "cubatao-p19" "cubatao-p14" ...
 $ q03: Named num [1:12] 0 0 1 1 1 0 0 0 2 0 ...
  ..- attr(*, "names")= chr [1:12] "cubatao-p50" "guaruja-p14" "sao-vicente-sao-paulo-p10" "sao-vicente-sao-paulo-p2" ...
 $ q04: Named num [1:11] 1 0 0 0 0 2 0 0 1 0 ...
  ..- attr(*, "names")= chr [1:11] "guaruja-p20" "ponta-da-praia-p1" "praia-das-asturias-p2" "praia-grande-sao-paulo-p22" ...
 $ q05: Named num [1:12] 0 0 0 1 0 0 0 0 0 0 ...
  ..- attr(*, "names")= chr [1:12] "sao-vicente-sao-paulo-p21" "praia-grande-sao-paulo-p51" "praia-grande-sao-paulo-p43" "praia-grande-sao-paulo-p3" ...


[1] 5

q01 q02 q03 q04 q05 
  2   2   2   1   0

[1] "q01" "q02" "q03" "q04"

[1] TRUE

In [16]:
lsa <- function(ix, k) {                                          # k: quantas dimensoes guardar
  s <- svd(ix$w)                                                  # W = U Sigma V'
  U <- s$u[, 1:k, drop = FALSE]; rownames(U) <- ix$vocab          # termos x k
  D <- s$v[, 1:k, drop = FALSE] %*% diag(s$d[1:k], nrow = k)      # documentos x k: V_k Sigma_k
  rownames(D) <- colnames(ix$w)                                   # d1 ... d8 (ou os IDs do grupo)
  list(U = U, S = s$d[1:k], D = D)                                # o modelo
}

ranking_lsa <- function(consulta, ix, cfg, modelo) {               # modelo: o que lsa() devolve
  q0 <- vetor_consulta(preparar(consulta, cfg), ix$vocab, ix$idf_tfidf)  # texto -> pesos
  qk <- t(modelo$U) %*% q0                                         # dobra: k coordenadas
  s  <- apply(modelo$D, 1, function(d) cosseno(qk, d))             # um cosseno por LINHA
  sort(s, decreasing = TRUE)                                       # do maior ao menor
}

In [17]:
dim(ix$w); min(dim(ix$w))                         # termos x documentos
k_max <- min(20, min(dim(ix$w)))
m20 <- lsa(ix, k_max)                             # LSA com k = 20 (ou k_max)

q <- usar[1]                                      # a primeira consulta valida de trabalho
rl <- ranking_lsa(consultas[[q]], ix, cfg, m20)  # LSA
rb <- ranking_bm25(consultas[[q]], ix, cfg)      # BM25

# Compara o top-5 do BM25 e do LSA
rbind(bm25 = names(rb)[1:5], lsa = names(rl)[1:5])

# Documentos que SO o LSA trouxe no top-5
so_lsa <- names(rl)[1:5][!names(rl)[1:5] %in% names(rb)[1:5]]
docs[so_lsa]

[1] 3305  194

[1] 194

bm25,gonzaga-santos-p1,gonzaga-santos-p3,jose-menino-p4,gonzaga-santos-p7,gonzaga-santos-p2
lsa,praia-do-tombo-p2,praia-grande-sao-paulo-p4,aparecida-santos-p3,aparecida-santos-p4,praia-grande-sao-paulo-p48


praia-do-tombo-p2 
                                                                                               "A praia tem aproximadamente 900 metros de extensão e conta com 5 quiosques para refeições, mar relativamente bravo e vários \"buracos\" fazem ser a praia preferida dos surfistas da região. Vizinha da Praia das Astúrias, fica longe do burburinho da cidade. Com ótima infraestrutura e balneabilidade, a praia do tombo devido sua formação geológica possui fortes ondas, o que atrai inúmeros praticantes de surf, além de ser palco de diversos campeonatos de surfing nacional e internacional." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               praia-grande-sao-paulo-p4 
                                                                                                "Entre os principais atrativos turísticos encontram-se a orla marítima, a Fortaleza de Itaipu, o Complexo Cultural Palácio das Artes, a Área de Lazer Ézio Dall'Acqua (Portinho), as feiras permanentes de artesanato, as praças temáticas e os monumentos distribuídos ao longo da orla. Publicações especializadas também destacam Praia Grande como um dos principais destinos de lazer do litoral paulista, reunindo informações sobre praias, hospedagem e infraestrutura voltadas aos visitantes." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     aparecida-santos-p3 
"Localizado na Zona Leste da cidade, entre os bairros do Embaré e Ponta da Praia, e também entre os canais 5 e 6, abriga diversos edifícios e conjuntos habitacionais para todas as classes sociais. Entre esses conjuntos, destaca-se o conjunto habitacional Marechal Humberto de Alencar Castello Branco, conhecido por BNH (Banco Nacional de Habitação) da Aparecida entre os maiores de toda a América Latina, na área onde nos anos 20, existia um hipódromo. Alem deste, existe tambem o Conjunto Habitacional Martins Fontes, popularmente conhecido como Jaú, e o conjunto habitacional IAPI." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     aparecida-santos-p4 
                                                                                                                                                                                                                                                         "Oferece diversas opções de comércio e lazer, com grande shopping center, praia, cinemas, feiras, teatro, escolas técnicas e universidade. Há, também o SESC Aparecida, um grande complexo de lazer e educação. Seu teat

In [18]:
# Quantas palavras da consulta esses documentos possuem?
sapply(so_lsa, function(d) length(intersect(preparar(consultas[[q]], cfg), ix$tokens[[d]])))

# E qual foi o grau de relevancia dado pelo grupo?
qrels[[q]][so_lsa]

praia-do-tombo-p2  praia-grande-sao-paulo-p4 
                         0                          0 
       aparecida-santos-p3        aparecida-santos-p4 
                         0                          0 
praia-grande-sao-paulo-p48 
                         0

<NA> <NA> <NA> <NA> <NA> 
  NA   NA   NA   NA   NA

In [19]:
consultas[usar[1]]
g <- qrels[[usar[1]]]
g[c("gonzaga-santos-p1","gonzaga-santos-p3","jose-menino-p4","gonzaga-santos-p7","gonzaga-santos-p2")]
g[c("praia-do-tombo-p2","praia-grande-sao-paulo-p4","aparecida-santos-p3","aparecida-santos-p4","praia-grande-sao-paulo-p48")]
docs[c("gonzaga-santos-p1","jose-menino-p4")]

q01 
"lugar calmo no gonzaga"

gonzaga-santos-p1 gonzaga-santos-p3    jose-menino-p4 gonzaga-santos-p7 
                2                 1                 0                 2 
gonzaga-santos-p2 
                0

<NA> <NA> <NA> <NA> <NA> 
  NA   NA   NA   NA   NA

gonzaga-santos-p1 
                                                                                                                                                                                                                                                                "O Gonzaga é um bairro nobre da cidade de Santos. A Praça da Independência, situada no Gonzaga é um ponto de referência do bairro, local onde são feitas manifestações, apresentações e comemorações." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                        jose-menino-p4 
"Outro destaque é o Templo da Paróquia Anglicana de Todos os Santos All Saints´Church (1918) - construído em estilo neogótico inglês, com vitrais dos patronos das Ilhas Britânicas, além de sua arquitetura única no Estado de São Paulo, é um lugar convidativo para oração e contemplação, lugar predileto no roteiro de turismo religioso. Está, deste 2008, em processo de tombamento como Patrimônio Histórico do Município de Santos e do Estado de São Paulo."

In [20]:
rrf <- function(..., c = 60) {                         # ...: quantos rankings vierem
  rks <- list(...)                                     # os rankings, numa lista
  ids <- unique(unlist(lapply(rks, names)))            # todos os documentos que aparecem
  s <- sapply(ids, function(d)                         # para cada documento...
    sum(sapply(rks, function(r) {                      # ...soma sobre os rankings
      p <- which(names(r) == d)                        # a posicao dele neste ranking
      if (length(p)) 1 / (c + p) else 0 })))           # ausente: nao soma nada
  sort(s, decreasing = TRUE)                           # do maior ao menor
}

ranking_modelo <- function(txt, modelo, ix, cfg, m, c = 60) {    # m: o que lsa() devolve
  rb <- ranking_bm25(txt, ix, cfg)                               # o lexico
  if (modelo == "bm25") return(rb)                               # so ele
  rl <- ranking_lsa(txt, ix, cfg, m)                             # o LSA
  if (modelo == "lsa") rl else rrf(rb, rl, c = c)                # so ele, ou a fusao RRF
}

ap_modelo <- function(q, modelo, m, consultas, qrels, ix, cfg)   # o AP de um modelo numa consulta
  metricas(ranking_modelo(consultas[[q]], modelo, ix, cfg, m), qrels[[q]], cfg$limiar)[["AP"]]

In [28]:
ks <- c(10, 20, 50)
ks <- ks[ks <= min(dim(ix$w))]
ms <- setNames(lapply(ks, function(k) lsa(ix, k)), paste0("k", ks))

nec$conjunto <- ifelse(nec$consulta %in% c("q01", "q02"), "dev", "teste")   # <- faltava

dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])
teste <- intersect(usar, nec$consulta[nec$conjunto == "teste"])
stopifnot(length(dev) > 0)   # para de forma clara se dev voltar vazio

tab_dev <- sapply(dev, function(q) c(
  bm25 = ap_modelo(q, "bm25", NULL, consultas, qrels, ix, cfg),
  lsa  = sapply(ms, function(m) ap_modelo(q, "lsa", m, consultas, qrels, ix, cfg)),
  hib  = sapply(ms, function(m) ap_modelo(q, "hibrido", m, consultas, qrels, ix, cfg))))

round(tab_dev, 3); round(rowMeans(tab_dev), 3)

,q01,q02
bm25,0.750,1.000
lsa.k10,0.022,0.500
lsa.k20,0.061,1.000
lsa.k50,0.106,1.000
hib.k10,0.101,0.833
hib.k20,0.600,1.000
hib.k50,0.268,1.000


bm25 lsa.k10 lsa.k20 lsa.k50 hib.k10 hib.k20 hib.k50 
  0.875   0.261   0.531   0.553   0.467   0.800   0.634

In [29]:
colnames(tab_dev)[tab_dev["hib.k20", ] < tab_dev["bm25", ]]

[1] "q01"

In [30]:
escolha <- "hibrido"  # troquem para a escolha de voces: "bm25", "lsa" ou "hibrido"
k_esc   <- 20         # troquem para o k escolhido de voces

m_esc <- lsa(ix, k_esc)
tab_teste <- sapply(teste, function(q) c(
  bm25      = ap_modelo(q, "bm25", NULL, consultas, qrels, ix, cfg),
  escolhido = ap_modelo(q, escolha, m_esc, consultas, qrels, ix, cfg)))

round(tab_teste, 3); round(rowMeans(tab_teste), 3)

,q03,q04
bm25,0.341,1
escolhido,0.600,1


bm25 escolhido 
     0.67      0.80

In [33]:
source("config.R")                                                     # recarrega as configuracoes
cfg$modelo; cfg$lsa_k; cfg$rrf_c                                       # confirma os parametros

ranking_motor <- function(txt, ix, cfg)                                # o ranking definitivo do motor
  ranking_modelo(txt, cfg$modelo, ix, cfg, lsa(ix, cfg$lsa_k), c = cfg$rrf_c)

# Testa o top-5 que o motor entrega para a primeira consulta valida
round(head(ranking_motor(consultas[[usar[1]]], ix, cfg), 5), 4)

[1] "hibrido"

[1] 20

[1] 60

gonzaga-santos-p7         gonzaga-santos-p3                cubatao-p8 
                   0.0293                    0.0288                    0.0286 
        gonzaga-santos-p2 praia-grande-sao-paulo-p9 
                   0.0285                    0.0284

In [34]:
anexar_estado("00_FICHA_PROJETO.md")            # o estado do R vai para a ficha
anexar_estado("aula07_parteD_consolidado.md")   # e para o consolidado do grupo

Estado do R anexado a 00_FICHA_PROJETO.md (64 linhas de objetos).

Estado do R anexado a aula07_parteD_consolidado.md (64 linhas de objetos).

